# Real-Time Network Monitoring: PySpark MLlib Distributed Offline Training

This notebook trains a native **PySpark MLlib RandomForestClassifier** pipeline using the network traffic Parquet files in the `data/` directory. The trained model is saved as a Spark `PipelineModel` for native distributed real-time prediction in the streaming job.

In [8]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
import json

# Create local Spark Session and limit to 2 execution threads to save memory
spark = SparkSession.builder \
    .master("local[2]") \
    .appName("MLlib_Offline_Training") \
    .config("spark.driver.memory", "4g") \
    .config("spark.hadoop.fs.defaultFS", "file:///") \
    .config("spark.sql.parquet.enableVectorizedReader", "true") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark Session Initialized.")

Spark Session Initialized.


In [9]:
# Dynamically resolve paths based on current working directory
cwd = os.getcwd()
if os.path.basename(cwd) == "training_model":
    project_root = os.path.abspath(os.path.join(cwd, ".."))
    model_dir = cwd
else:
    project_root = cwd
    model_dir = os.path.abspath(os.path.join(cwd, "training_model"))

data_path = os.path.join(project_root, "data")
model_save_path = os.path.join(model_dir, "spark_rf_model")
report_save_path = os.path.join(model_dir, "spark_evaluation_report.json")

print(f"Project root resolved to: {project_root}")
print(f"Data path resolved to: {data_path}")
print(f"Model save path resolved to: {model_save_path}")
print(f"Report save path resolved to: {report_save_path}")

FEATURE_COLS = [
    'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 
    'Fwd Packets Length Total', 'Bwd Packets Length Total', 'Fwd Packet Length Max', 
    'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 
    'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 
    'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 
    'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 
    'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 
    'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 
    'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 
    'Fwd Packets/s', 'Bwd Packets/s', 'Packet Length Min', 'Packet Length Max', 
    'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 
    'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 
    'ACK Flag Count', 'URG Flag Count', 'CWE Flag Count', 'ECE Flag Count', 
    'Down/Up Ratio', 'Avg Packet Size', 'Avg Fwd Segment Size', 'Avg Bwd Segment Size', 
    'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 
    'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate', 
    'Subflow Fwd Packets', 'Subflow Fwd Bytes', 'Subflow Bwd Packets', 
    'Subflow Bwd Bytes', 'Init Fwd Win Bytes', 'Init Bwd Win Bytes', 
    'Fwd Act Data Packets', 'Fwd Seg Size Min', 'Active Mean', 'Active Std', 
    'Active Max', 'Active Min', 'Idle Mean', 'Idle Std', 'Idle Max', 'Idle Min'
]

# Find all parquet files in data_path
parquet_files = [os.path.join(data_path, f) for f in os.listdir(data_path) if f.endswith(".parquet")]
print(f"Found {len(parquet_files)} parquet files to load.")

# Load, cast, and accumulate dataframes
dfs = []
for file in parquet_files:
    file_df = spark.read.parquet(file)
    
    # Map target column
    file_df = file_df.withColumn("target", F.when(F.col("Label") == "Benign", 0.0).otherwise(1.0))
    
    # Filter out infinite values (common in CICIDS2017 flow rates)
    for col_name in ["Flow Bytes/s", "Flow Packets/s"]:
        if col_name in file_df.columns:
            file_df = file_df.filter((F.col(col_name) != float('inf')) & (F.col(col_name) != float('-inf')))

    # Select features and cast to Double
    select_exprs = [F.col(c).cast("double").alias(c) for c in FEATURE_COLS] + [F.col("target")]
    clean_file_df = file_df.select(*select_exprs).na.fill(0.0)
    
    # Sample 10% (approx 250,000 rows, highly representative and prevents local OOM)
    sampled_df = clean_file_df.sample(fraction=0.1, seed=42)
    dfs.append(sampled_df)

# Union all dataframes together
from functools import reduce
df_raw_union = reduce(lambda df1, df2: df1.union(df2), dfs)

# STAGING: Write to a temp parquet folder with reduced parallelism (2 partitions) to free memory
temp_stage_path = os.path.join(model_dir, "temp_clean_stage")
print("Checkpointing cleaned data to disk...")
df_raw_union.coalesce(2).write.mode("overwrite").parquet(temp_stage_path)

# Load the staged data back
df_clean = spark.read.parquet(temp_stage_path)

print("Data loaded and staged successfully.")
print(f"Total features configured: {len(FEATURE_COLS)}")

Project root resolved to: /mnt/f/Tin/code/UETTTTTTTTTTTTTTTTTTTTTTT/Big Data/Network-Monitoring
Data path resolved to: /mnt/f/Tin/code/UETTTTTTTTTTTTTTTTTTTTTTT/Big Data/Network-Monitoring/data
Model save path resolved to: /mnt/f/Tin/code/UETTTTTTTTTTTTTTTTTTTTTTT/Big Data/Network-Monitoring/training_model/spark_rf_model
Report save path resolved to: /mnt/f/Tin/code/UETTTTTTTTTTTTTTTTTTTTTTT/Big Data/Network-Monitoring/training_model/spark_evaluation_report.json
Found 7 parquet files to load.
Checkpointing cleaned data to disk...


Data loaded and staged successfully.
Total features configured: 76


In [10]:
# Show class distribution
df_clean.groupBy("target").count().show(30, truncate=False)

+------+------+
|target|count |
+------+------+
|0.0   |158030|
|1.0   |14135 |
+------+------+



In [11]:
# Assemble features in exact order
assembler = VectorAssembler(inputCols=FEATURE_COLS, outputCol="features", handleInvalid="keep")

# Standardize features
scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures", withStd=True, withMean=False)

# Define Random Forest Classifier (Optimized for local resources)
rf = RandomForestClassifier(
    featuresCol="scaledFeatures", 
    labelCol="target", 
    numTrees=20, 
    maxDepth=8, 
    seed=42
)

# Build Spark ML Pipeline
pipeline = Pipeline(stages=[assembler, scaler, rf])

In [12]:
# Split the dataset (80% train, 20% test)
train_df, test_df = df_clean.randomSplit([0.8, 0.2], seed=42)

print("Starting Distributed training using Spark MLlib...")
# Fit the model
model = pipeline.fit(train_df)
print("Training completed successfully!")

Starting Distributed training using Spark MLlib...


ERROR:root:Exception while sending command.                         (0 + 2) / 2]
Traceback (most recent call last):
  File "/mnt/f/Tin/code/UETTTTTTTTTTTTTTTTTTTTTTT/Big Data/Network-Monitoring/venv/lib/python3.12/site-packages/py4j/clientserver.py", line 516, in send_command
    raise Py4JNetworkError("Answer from Java side is empty")
py4j.protocol.Py4JNetworkError: Answer from Java side is empty

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/mnt/f/Tin/code/UETTTTTTTTTTTTTTTTTTTTTTT/Big Data/Network-Monitoring/venv/lib/python3.12/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/mnt/f/Tin/code/UETTTTTTTTTTTTTTTTTTTTTTT/Big Data/Network-Monitoring/venv/lib/python3.12/site-packages/py4j/clientserver.py", line 539, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while sen

Py4JError: An error occurred while calling o4093.fit

In [ ]:
# Predict on test set
predictions = model.transform(test_df)

# Evaluate using MulticlassClassificationEvaluator
evaluator_acc = MulticlassClassificationEvaluator(labelCol="target", predictionCol="prediction", metricName="accuracy")
evaluator_f1 = MulticlassClassificationEvaluator(labelCol="target", predictionCol="prediction", metricName="f1")

accuracy = evaluator_acc.evaluate(predictions)
f1_score = evaluator_f1.evaluate(predictions)

print(f"Test Accuracy: {accuracy:.4f}")
print(f"Test F1-Score: {f1_score:.4f}")

# Save evaluation report to JSON
report = {
    "model_type": "PySpark_MLlib_RandomForest",
    "num_trees": 20,
    "max_depth": 8,
    "test_accuracy": accuracy,
    "test_f1_score": f1_score
}

with open(report_save_path, "w") as f:
    json.dump(report, f, indent=4)
print(f"Evaluation report saved to {report_save_path}")

## Side-by-Side Model Comparison (Scikit-Learn vs PySpark MLlib)

We load the old Scikit-Learn Random Forest model (`rf_pipeline.pkl`) and evaluate both models on the same validation data subset to check for any performance drift.

In [ ]:
import joblib
from sklearn.metrics import accuracy_score, f1_score

print("Converting test subset to Pandas for Scikit-Learn validation...")
test_pdf = test_df.toPandas()
X_test = test_pdf[FEATURE_COLS]
y_test = test_pdf['target']

# Resolve path to the old Scikit-Learn model
sklearn_model_path = os.path.join(model_dir, "rf_pipeline.pkl")
if not os.path.exists(sklearn_model_path):
    sklearn_model_path = os.path.join(project_root, "rf_pipeline.pkl")

print(f"Loading Scikit-Learn model from: {sklearn_model_path}")
sklearn_model = joblib.load(sklearn_model_path)

# Predict with Scikit-Learn model
y_pred_sklearn = sklearn_model.predict(X_test)
sklearn_acc = accuracy_score(y_test, y_pred_sklearn)
sklearn_f1 = f1_score(y_test, y_pred_sklearn)

print("\n=== Side-by-Side Model Comparison ===")
print(f"Scikit-Learn Model  - Accuracy: {sklearn_acc:.4f}, F1-Score: {sklearn_f1:.4f}")
print(f"PySpark MLlib Model - Accuracy: {accuracy:.4f}, F1-Score: {f1_score:.4f}")

In [ ]:
# Save the Spark MLlib PipelineModel
model.write().overwrite().save(model_save_path)
print(f"Spark MLlib PipelineModel saved successfully to: {model_save_path}")